### Salida multimodal: texto → imagen / texto → voz

Hasta ahora el LLM devolvía solo texto. "Multimodal" acá significa que la salida puede ser imagen o audio en vez de (o además de) texto — DALL·E genera imágenes desde una descripción, TTS convierte texto en voz. Para el Proyecto 2 (agente de aerolínea), esto es lo que deja que el agente muestre una foto del destino o responda hablando en vez de solo escribir.

El curso original usa DALL·E/TTS de OpenAI — acá no hay `OPENAI_API_KEY` cargada, así que usamos los equivalentes de Gemini.

**Generación de imágenes — bloqueada en este proyecto.** El modelo de imagen de Gemini (`gemini-3-pro-image-preview`) devuelve `429 RESOURCE_EXHAUSTED` con `limit: 0` — no es que se gastó la cuota del día, es que el **free tier no tiene acceso** a generación de imágenes en absoluto. Haría falta cuenta paga (o volver a probar con `OPENAI_API_KEY` si se consigue). Queda documentado como límite real del entorno, no como error a resolver.

In [4]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents="Hola, este es un vuelo a Roma, tendremos turbulencia a las tres y media. alerta.",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
            )
        ),
    ),
)
audio_part = response.candidates[0].content.parts[0].inline_data
print("Audio generado:", len(audio_part.data), "bytes —", audio_part.mime_type)


Audio generado: 309646 bytes — audio/L16;codec=pcm;rate=24000


El audio viene como PCM crudo, sin cabecera de archivo — hay que envolverlo en formato WAV antes de poder reproducirlo o guardarlo como `.wav`. Usamos el módulo estándar `wave` de Python.

In [5]:
import wave
from IPython.display import Audio

with wave.open("respuesta.wav", "wb") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)
    wav_file.setframerate(24000)
    wav_file.writeframes(audio_part.data)

Audio("respuesta.wav")
